# Twenty Questions Experiment Analysis

This notebook analyzes results from experiments produced by `fixed_game_loop.py`.

**Input:** `run_dir` - a directory in `experiments/` (e.g., `experiments/run_2025_11_17_210902`)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Configuration

Set the `run_dir` to the experiment directory you want to analyze:

In [ ]:
# Set the experiment directory to analyze
run_dir = (
    "experiments/run_2025_11_17_214324"  # Change this to your experiment directory
)

# Verify directory exists
if not os.path.exists(run_dir):
    raise ValueError(f"Directory {run_dir} does not exist")

print(f"Analyzing experiment: {run_dir}")

## 2. Load Data

In [ ]:
# Load summary.json
summary_path = os.path.join(run_dir, 'summary.json')
with open(summary_path, 'r') as f:
    summary_data = json.load(f)

# Extract experiment metadata
experiment_meta = summary_data['experiment_summary']
print("\n=== Experiment Metadata ===")
print(f"Total games: {experiment_meta['total_games']}")
print(f"Completed games: {experiment_meta['completed_games']}")
print(f"Failed games: {experiment_meta['failed_games']}")
print(f"Models tested: {experiment_meta['models_tested']}")
print(f"Agent types tested: {experiment_meta['agent_types_tested']}")
print(f"Games per model per agent: {experiment_meta['games_per_model_per_agent']}")
print(f"Total duration: {experiment_meta['total_duration']:.1f} seconds ({experiment_meta['total_duration']/60:.1f} minutes)")

In [ ]:
# Create summary DataFrame from summary.json results
summary_df = pd.DataFrame(summary_data['results'])
print(f"\nSummary DataFrame shape: {summary_df.shape}")
summary_df.head()

In [ ]:
# Load all individual game JSONs from games/ subdirectory
games_dir = os.path.join(run_dir, 'games')
game_files = list(Path(games_dir).glob('*.json'))

print(f"Found {len(game_files)} game files")

# Load all games into a list
games = []
for game_file in game_files:
    with open(game_file, 'r') as f:
        games.append(json.load(f))

# Create games DataFrame
game_df = pd.DataFrame(games)
print(f"\nGames DataFrame shape: {game_df.shape}")
game_df.head()

## 3. Data Processing

Extract key metrics from the game data:

In [ ]:
# Filter out failed games (games with 'error' field)
successful_games = game_df[~game_df['error'].notna()].copy() if 'error' in game_df.columns else game_df.copy()

print(f"Successful games: {len(successful_games)} / {len(game_df)}")

# Extract win status from rewards (reward of 1 means win)
successful_games['won'] = successful_games['rewards'].apply(lambda x: x.get('0', 0) == 1 if isinstance(x, dict) else False)

# Extract game reason
successful_games['outcome_reason'] = successful_games['game_info'].apply(
    lambda x: x.get('0', {}).get('reason', 'Unknown') if isinstance(x, dict) else 'Unknown'
)

print(f"\nWin rate: {successful_games['won'].mean():.2%}")
print(f"Average turns: {successful_games['turn_count'].mean():.2f}")
print(f"Average duration: {successful_games['game_duration'].mean():.2f} seconds")

## 4. Basic Analysis

### 4.1 Win Rates by Agent Type and Model

In [ ]:
# Win rates by agent type
win_rate_by_agent = successful_games.groupby('agent_type')['won'].agg(['mean', 'count', 'sum'])
win_rate_by_agent.columns = ['win_rate', 'total_games', 'wins']
win_rate_by_agent = win_rate_by_agent.sort_values('win_rate', ascending=False)

print("\n=== Win Rates by Agent Type ===")
print(win_rate_by_agent)

# Win rates by model
win_rate_by_model = successful_games.groupby('model_name')['won'].agg(['mean', 'count', 'sum'])
win_rate_by_model.columns = ['win_rate', 'total_games', 'wins']
win_rate_by_model = win_rate_by_model.sort_values('win_rate', ascending=False)

print("\n=== Win Rates by Model ===")
print(win_rate_by_model)

# Win rates by both agent type and model
win_rate_by_both = successful_games.groupby(['model_name', 'agent_type'])['won'].agg(['mean', 'count', 'sum'])
win_rate_by_both.columns = ['win_rate', 'total_games', 'wins']
win_rate_by_both = win_rate_by_both.sort_values('win_rate', ascending=False)

print("\n=== Win Rates by Model and Agent Type ===")
print(win_rate_by_both)

In [ ]:
# Visualize win rates
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Win rate by agent type
ax1 = axes[0]
win_rate_by_agent['win_rate'].plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Win Rate by Agent Type', fontsize=14, fontweight='bold')
ax1.set_xlabel('Agent Type', fontsize=12)
ax1.set_ylabel('Win Rate', fontsize=12)
ax1.set_ylim(0, 1)
ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='50% baseline')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, v in enumerate(win_rate_by_agent['win_rate']):
    ax1.text(i, v + 0.02, f'{v:.1%}', ha='center', va='bottom', fontweight='bold')

# Win rate by model
ax2 = axes[1]
win_rate_by_model['win_rate'].plot(kind='bar', ax=ax2, color='coral')
ax2.set_title('Win Rate by Model', fontsize=14, fontweight='bold')
ax2.set_xlabel('Model', fontsize=12)
ax2.set_ylabel('Win Rate', fontsize=12)
ax2.set_ylim(0, 1)
ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='50% baseline')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, v in enumerate(win_rate_by_model['win_rate']):
    ax2.text(i, v + 0.02, f'{v:.1%}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart: Win rate by model and agent type
win_rate_pivot = successful_games.pivot_table(
    values='won',
    index='model_name',
    columns='agent_type',
    aggfunc='mean'
)

ax = win_rate_pivot.plot(kind='bar', figsize=(12, 6), width=0.8)
ax.set_title('Win Rate by Model and Agent Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Win Rate', fontsize=12)
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='50% baseline')
ax.legend(title='Agent Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.2 Number of Turns by Agent Type and Model

In [ ]:
# Average turns by agent type
turns_by_agent = successful_games.groupby('agent_type')['turn_count'].agg(['mean', 'std', 'count'])
turns_by_agent.columns = ['avg_turns', 'std_turns', 'count']
turns_by_agent = turns_by_agent.sort_values('avg_turns')

print("\n=== Average Turns by Agent Type ===")
print(turns_by_agent)

# Average turns by model
turns_by_model = successful_games.groupby('model_name')['turn_count'].agg(['mean', 'std', 'count'])
turns_by_model.columns = ['avg_turns', 'std_turns', 'count']
turns_by_model = turns_by_model.sort_values('avg_turns')

print("\n=== Average Turns by Model ===")
print(turns_by_model)

# Average turns by both agent type and model
turns_by_both = successful_games.groupby(['model_name', 'agent_type'])['turn_count'].agg(['mean', 'std', 'count'])
turns_by_both.columns = ['avg_turns', 'std_turns', 'count']
turns_by_both = turns_by_both.sort_values('avg_turns')

print("\n=== Average Turns by Model and Agent Type ===")
print(turns_by_both)

In [ ]:
# Visualize turn counts
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Average turns by agent type
ax1 = axes[0]
turns_by_agent['avg_turns'].plot(kind='bar', ax=ax1, color='steelblue', yerr=turns_by_agent['std_turns'])
ax1.set_title('Average Turns by Agent Type', fontsize=14, fontweight='bold')
ax1.set_xlabel('Agent Type', fontsize=12)
ax1.set_ylabel('Average Turns', fontsize=12)
ax1.grid(axis='y', alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, v in enumerate(turns_by_agent['avg_turns']):
    ax1.text(i, v + 0.5, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# Average turns by model
ax2 = axes[1]
turns_by_model['avg_turns'].plot(kind='bar', ax=ax2, color='coral', yerr=turns_by_model['std_turns'])
ax2.set_title('Average Turns by Model', fontsize=14, fontweight='bold')
ax2.set_xlabel('Model', fontsize=12)
ax2.set_ylabel('Average Turns', fontsize=12)
ax2.grid(axis='y', alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, v in enumerate(turns_by_model['avg_turns']):
    ax2.text(i, v + 0.5, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart: Average turns by model and agent type
turns_pivot = successful_games.pivot_table(
    values='turn_count',
    index='model_name',
    columns='agent_type',
    aggfunc='mean'
)

ax = turns_pivot.plot(kind='bar', figsize=(12, 6), width=0.8)
ax.set_title('Average Turns by Model and Agent Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Average Turns', fontsize=12)
ax.legend(title='Agent Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Box plot: Turn count distribution by agent type
fig, ax = plt.subplots(figsize=(12, 6))
successful_games.boxplot(column='turn_count', by='agent_type', ax=ax)
ax.set_title('Turn Count Distribution by Agent Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Agent Type', fontsize=12)
ax.set_ylabel('Turn Count', fontsize=12)
plt.suptitle('')  # Remove the default title
plt.tight_layout()
plt.show()

### 4.3 Additional Analysis: Win Rate vs Turn Count

In [ ]:
# Analyze relationship between turns and winning
print("\n=== Turns Analysis by Outcome ===")
print("\nWinning games:")
print(successful_games[successful_games['won']]['turn_count'].describe())
print("\nLosing games:")
print(successful_games[~successful_games['won']]['turn_count'].describe())

In [ ]:
# Distribution of turn counts for wins vs losses
fig, ax = plt.subplots(figsize=(12, 6))

successful_games[successful_games['won']]['turn_count'].hist(
    bins=20, alpha=0.6, label='Wins', ax=ax, color='green'
)
successful_games[~successful_games['won']]['turn_count'].hist(
    bins=20, alpha=0.6, label='Losses', ax=ax, color='red'
)

ax.set_title('Turn Count Distribution: Wins vs Losses', fontsize=14, fontweight='bold')
ax.set_xlabel('Turn Count', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 4.4 Game Duration Analysis

In [ ]:
# Average duration by agent type and model
duration_by_agent = successful_games.groupby('agent_type')['game_duration'].agg(['mean', 'std', 'count'])
duration_by_agent.columns = ['avg_duration', 'std_duration', 'count']
duration_by_agent = duration_by_agent.sort_values('avg_duration')

print("\n=== Average Duration (seconds) by Agent Type ===")
print(duration_by_agent)

duration_by_model = successful_games.groupby('model_name')['game_duration'].agg(['mean', 'std', 'count'])
duration_by_model.columns = ['avg_duration', 'std_duration', 'count']
duration_by_model = duration_by_model.sort_values('avg_duration')

print("\n=== Average Duration (seconds) by Model ===")
print(duration_by_model)

## 5. Summary Statistics Table

In [ ]:
# Create comprehensive summary table
summary_stats = successful_games.groupby(['model_name', 'agent_type']).agg({
    'won': ['mean', 'sum', 'count'],
    'turn_count': ['mean', 'std', 'min', 'max'],
    'game_duration': ['mean', 'std']
}).round(2)

summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]
summary_stats = summary_stats.rename(columns={
    'won_mean': 'win_rate',
    'won_sum': 'wins',
    'won_count': 'total_games',
    'turn_count_mean': 'avg_turns',
    'turn_count_std': 'std_turns',
    'turn_count_min': 'min_turns',
    'turn_count_max': 'max_turns',
    'game_duration_mean': 'avg_duration_sec',
    'game_duration_std': 'std_duration_sec'
})

print("\n=== Comprehensive Summary Statistics ===")
print(summary_stats.sort_values('win_rate', ascending=False))

In [ ]:
# Export summary to CSV
output_csv = os.path.join(run_dir, 'analysis_summary.csv')
summary_stats.to_csv(output_csv)
print(f"\nSummary statistics exported to: {output_csv}")